In [ ]:
import os
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import kagglehub
import shap

def load_dataset(base_path, max_samples_per_class=100, random_seed=42):
    """
    Load a balanced subset of images from the dataset
    """
    np.random.seed(random_seed)
    processed_images = []
    original_images = []
    paths = []
    labels = []

    # Get aneurysm files
    aneurysm_path = os.path.join(base_path, 'aneurysm')
    if os.path.exists(aneurysm_path):
        aneurysm_files = [(os.path.join(aneurysm_path, f), 1)
                         for f in os.listdir(aneurysm_path)
                         if f.endswith(('.jpg', '.dcm'))]

    # Check for non-aneurysm images
    non_aneurysm_files = []
    for category in ['tumor', 'cancer']:
        category_path = os.path.join(base_path, category)
        if os.path.exists(category_path):
            non_aneurysm_files.extend([(os.path.join(category_path, f), 0)
                                        for f in os.listdir(category_path)
                                        if f.endswith(('.jpg', '.dcm'))])

    # Randomly sample files
    n_aneurysm = min(len(aneurysm_files), max_samples_per_class)
    n_other = min(len(non_aneurysm_files), max_samples_per_class)

    selected_aneurysm_idx = np.random.choice(np.arange(len(aneurysm_files)), n_aneurysm, replace=False)
    selected_other_idx = np.random.choice(np.arange(len(non_aneurysm_files)), n_other, replace=False)

    selected_files = [aneurysm_files[i] for i in selected_aneurysm_idx]
    selected_files.extend([non_aneurysm_files[i] for i in selected_other_idx])

    # Load selected files
    print(f"Loading {len(selected_files)} images...")
    for file_path, label in selected_files:
        processed_img, original_img = load_and_preprocess_single_image(file_path)

        if processed_img is not None:
            processed_images.append(processed_img)
            original_images.append(original_img)
            paths.append(file_path)
            labels.append(label)

    if len(processed_images) == 0:
        raise ValueError(f"No valid images found in {base_path}")

    return (np.array(processed_images),
            np.array(original_images),
            np.array(paths),
            np.array(labels))

def load_and_preprocess_single_image(img_path):
    """Load and preprocess a single image for prediction"""
    try:
        if img_path.lower().endswith('.dcm'):
            # Handle DICOM files
            dcm = pydicom.dcmread(img_path)
            image = dcm.pixel_array.astype(float)
            window_center = 40
            window_width = 80
            min_value = window_center - window_width // 2
            max_value = window_center + window_width // 2
            image = np.clip(image, min_value, max_value)
            if len(image.shape) == 2:
                image = np.stack([image] * 3, axis=-1)
        else:
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError(f"Failed to load image: { img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Normalize to [0,1]
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
        image = image.astype(np.float32)

        # Resize for model input
        processed_img = cv2.resize(image, (224, 224))
        return processed_img, image  # Return both processed and original images

    except Exception as e:
        print(f"Error processing image {img_path}: {str(e)}")
        return None, None

def create_model(model_name='efficientnet'):
    """Create a model for medical image classification with specified architecture"""
    if model_name.lower() == 'efficientnet':
        base_model = EfficientNetB0(
            include_top=False,
            weights='imagenet',
            input_shape=(224, 224, 3)
        )
        inputs = layers.Input(shape=(224, 224, 3))
        x = base_model(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation='relu')(x)
        outputs = layers.Dense(1, activation='sigmoid')(x)  # Binary classification
        model = tf.keras.Model(inputs, outputs)
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

def train_model(model, train_images, train_labels, val_images, val_labels, epochs=10, batch_size=32):
    """Train the model on the training dataset"""
    history = model.fit(train_images, train_labels,
                        validation_data=(val_images, val_labels),
                        epochs=epochs,
                        batch_size=batch_size)
    return history

def evaluate_model(model, test_images, test_labels):
    """Evaluate the model on the test dataset"""
    loss, accuracy = model.evaluate(test_images, test_labels)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

def generate_shap_explanations(model, images, background_images):
    """Generate SHAP explanations for the model predictions"""
    # Use DeepExplainer for Keras models
    explainer = shap.DeepExplainer(model, background_images)
    shap_values = explainer.shap_values(images)

    # Visualize the SHAP values
    shap.summary_plot(shap_values, images)

def main():
    dataset_path = kagglehub.dataset_download("trainingdatapro/computed-tomography-ct-of-the-brain")
    base_path = os.path.join(dataset_path, 'files')

    # Load dataset
    processed_images, original_images, paths, labels = load_dataset(base_path)

    # Split dataset into training, validation, and test sets
    train_images, temp_images, train_labels, temp_labels = train_test_split(processed_images, labels, test_size=0.3, random_state=42)
    val_images, test_images, val_labels, test_labels = train_test_split(temp_images, temp_labels, test_size=0.5, random_state=42)

    # Create and train model
    model = create_model()
    train_model(model, train_images, train_labels, val_images, val_labels)

    # Evaluate model
    evaluate_model(model, test_images, test_labels)

    # Generate SHAP explanations using a subset of training images as background
    background_images = train_images[:100]  # Use a subset for background
    generate_shap_explanations(model, test_images, background_images)

if __name__ == "__main__":
    main()

100%|██████████| 66.0M/66.0M [00:00<00:00, 159MB/s]

Extracting files...


Loading 200 images...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 95s 9s/step - accuracy: 0.6630 - loss: 0.5355 - val_accuracy: 0.5333 - val_loss: 0.7570
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 75s 7s/step - accuracy: 0.9519 - loss: 0.0971 - val_accuracy: 0.5333 - val_loss: 0.8741
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 37s 7s/step - accuracy: 0.9896 - loss: 0.0357 - val_accuracy: 0.5333 - val_loss: 0.7388
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 36s 7s/step - accuracy: 0.9655 - loss: 0.0546 - val_accuracy: 0.5333 - val_loss: 0.7742
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 41s 7s/step - accuracy: 0.9770 - loss: 0.0479 - val_accuracy: 0.5333 - val_loss: 0.8179
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 42s 7s/step - accuracy: 0.9868 - loss: 0.0217 - val_accuracy: 0.5333 - val_loss: 0.8912
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 37s 7s/step - accuracy: 0.9588 - loss: 0.1270 - val_accuracy: 0.5333 - val_loss: 0.8487
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 42s 7s/step - accuracy:

/usr/local/lib/python3.11/dist-packages/shap/explainers/_deep/deep_tf.py:94: UserWarning: Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_238
Received: inputs=['Tensor(shape=(100, 224, 224, 3))']
  warnings.warn(msg)


In [ ]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.4 MB/s eta 0:00:00
